In [ ]:
# Se importan las librerías usadas para verificar y documentar KPI.

from pathlib import Path
import sqlite3

import pandas as pd
import plotly.express as px

In [ ]:
# Se definen la fuente del mart y los reportes de gobierno de KPI.

mart_file = Path("../data/sales_mart.db")
submission_directory = Path("../submission")
submission_directory.mkdir(exist_ok=True)

In [ ]:
# Se carga la medida de ventas con el grano de hecho para validar sus reglas.

with sqlite3.connect(mart_file) as connection:
    fact_sales = pd.read_sql_query("SELECT * FROM fact_sales", connection)
    dim_date = pd.read_sql_query("SELECT * FROM dim_date", connection)
fact_sales.shape, dim_date.shape

In [ ]:
# Se define el catálogo mínimo de indicadores que podrá usar un reporte gerencial.

kpi_catalog = pd.DataFrame([
    ["Ventas netas", "SUM(net_sales)", "No aplica", "Línea de pedido", "Mensual", "Gerencia comercial", "Mart de ventas"],
    ["Unidades vendidas", "SUM(quantity)", "No aplica", "Línea de pedido", "Mensual", "Gerencia comercial", "Mart de ventas"],
    ["Descuento promedio ponderado", "SUM(gross_sales * discount_pct)", "SUM(gross_sales)", "Línea de pedido", "Mensual", "Gerencia comercial", "Mart de ventas"],
], columns=["kpi", "numerador_o_formula", "denominador", "grano", "período", "propietario", "fuente"])
kpi_catalog

In [ ]:
# Se verifican reglas que impedirían publicar indicadores poco confiables.

quality_checks = pd.DataFrame([
    ["Llave de hecho única", not fact_sales[["order_id", "line_id"]].duplicated().any()],
    ["Ventas netas no negativas", fact_sales["net_sales"].ge(0).all()],
    ["Cantidades positivas", fact_sales["quantity"].gt(0).all()],
    ["Fechas del hecho presentes", fact_sales["date_key"].isin(dim_date["date_key"]).all()],
], columns=["regla", "pasa"])
quality_checks["estado"] = quality_checks["pasa"].map({True: "PASS", False: "FAIL"})
quality_checks

In [ ]:
# Se documenta la trazabilidad desde el hecho hasta los KPI publicados.

metric_lineage = pd.DataFrame([
    ["fact_sales.net_sales", "Ventas netas", "SUM por período y segmento"],
    ["fact_sales.quantity", "Unidades vendidas", "SUM por período y segmento"],
    ["fact_sales.gross_sales y discount_pct", "Descuento promedio ponderado", "SUM descuento / SUM venta bruta"],
], columns=["campo_origen", "kpi", "transformación"])
metric_lineage

In [ ]:
# ¿Podemos publicar estos KPI sin esconder problemas de calidad o definición?

publication_decision = pd.DataFrame({
    "decisión": ["Publicar catálogo de KPI"],
    "estado": ["APROBADO" if quality_checks["pasa"].all() else "BLOQUEADO"],
    "razón": ["Todas las reglas de calidad pasan." if quality_checks["pasa"].all() else "Existe al menos una regla fallida."],
})
publication_decision

In [ ]:
# Se publican catálogo, controles y trazabilidad como entregables de gobierno.

kpi_catalog.to_csv(submission_directory / "kpi_catalog.csv", index=False)
quality_checks.to_csv(submission_directory / "kpi_quality_report.csv", index=False)
metric_lineage.to_csv(submission_directory / "metric_lineage.csv", index=False)
publication_decision.to_csv(submission_directory / "kpi_publication_decision.csv", index=False)

In [ ]:
# Se visualiza el estado de los controles antes de publicar los indicadores.

fig = px.bar(
    quality_checks, x="regla", y="pasa", color="estado", text="estado",
    title="Controles de calidad para publicación de KPI",
    labels={"regla": "Regla", "pasa": "Control superado"},
    color_discrete_map={"PASS": "#4e79a7", "FAIL": "#e15759"},
)
fig.update_yaxes(tickvals=[0, 1], ticktext=["No", "Sí"])
fig.update_layout(template="plotly_white")
fig.show()